# Qwen2.5-VL — Évaluation pour le scan de cartes TCG

Banc de mesure de la sous-tâche **COLLR-534**.

Ce notebook **ne contient aucun résultat** tant qu'il n'a pas été exécuté.
Il *produit* les chiffres que tu reporteras ensuite dans `readme.md`.

## Ce qu'il mesure

| Critère imposé par `OCR/Models/readme.md` | Comment |
|---|---|
| Précision de l'OCR | comparaison champ par champ avec le corrigé `ground_truth.json` |
| Temps de scan par carte | `time.perf_counter()` autour de l'inférence, retry inclus |
| Consommation VRAM | `torch.cuda.max_memory_allocated()` — pic réel pendant l'inférence |

## Prérequis

1. Un GPU NVIDIA avec CUDA. Le modèle est chargé en int4, ~5 Go de VRAM.
2. **`ground_truth.json` rempli à la main.** Sans corrigé, le notebook mesure
   le temps et la VRAM, mais la précision reste incalculable.

## Périmètre — à lire avant de conclure

Ce banc évalue **3 champs** : `name`, `set_code`, `set_number`.

`docs/AI/model_evaluation/evaluation.md` §1 en demande davantage (HP, types,
attaques, rareté). Le prompt repris du prototype ne les extrait pas. Les
résultats valent donc pour **l'identification** d'une carte, pas pour
l'extraction complète de son contenu. C'est une limite à mentionner
explicitement dans le rapport.

## 1. Configuration

In [1]:
from pathlib import Path

# --- À ADAPTER SI BESOIN -----------------------------------------------------

# Dossier contenant les photos de cartes (prototype de Master 1).
IMAGES_DIR = Path(
    r"C:\Users\theor\Documents\Epitech Master 1er année\ESP CollectioneuR"
    r"\Scan Qwen2.5-VL-7B-AWQ\pokemon-card-scanner-qwen-7b\backend\uploads"
)

# Le corrigé, à côté de ce notebook.
GROUND_TRUTH = Path("ground_truth.json")

# Sortie brute : une ligne par carte, pour rejouer l'analyse sans relancer le GPU.
RESULTS_CSV = Path("resultats_qwen.csv")

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# Résolution envoyée au modèle. Repris du prototype : plus de pixels = meilleure
# lecture du petit texte du footer, mais plus de VRAM et de latence.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28

# -----------------------------------------------------------------------------

print(f"Images  : {IMAGES_DIR}  (existe : {IMAGES_DIR.exists()})")
print(f"Corrigé : {GROUND_TRUTH.resolve()}  (existe : {GROUND_TRUTH.exists()})")

Images  : C:\Users\theor\Documents\Epitech Master 1er année\ESP CollectioneuR\Scan Qwen2.5-VL-7B-AWQ\pokemon-card-scanner-qwen-7b\backend\uploads  (existe : True)
Corrigé : C:\Users\theor\Documents\Epitech Master 2ème année\ESP - CollectionR\Repo Projet\OCR\Models\Qwen_OCR\ground_truth.json  (existe : True)


## 2. Dépendances

À exécuter une seule fois. **`torch` doit être installé avant, avec CUDA.**

In [2]:
# torch AVANT tout le reste — adapter cu124 à ta version (vérifier avec `nvidia-smi`) :
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

# !pip install "transformers>=4.49.0" "accelerate>=1.0.0" "qwen-vl-utils[decord]>=0.0.8"
# !pip install "bitsandbytes>=0.43.0" pillow
# !pip install jiwer rapidfuzz

## 3. Chargement du modèle

Configuration reprise telle quelle du prototype `backend/app.py`.

In [3]:
import json, re, time, csv, statistics
import torch
from PIL import Image
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA indisponible. Le modèle est chargé en int4 sur GPU NVIDIA. "
        "Vérifie l'installation de torch avec CUDA."
    )

DEVICE = "cuda:0"
print(f"GPU : {torch.cuda.get_device_name(0)} "
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go VRAM)")

# nf4 (bitsandbytes) plutôt qu'AWQ : même empreinte finale (~5 Go) sans triton
# ni Visual Studio Build Tools sur Windows.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map=DEVICE,
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)

VRAM_POIDS_GO = torch.cuda.memory_allocated() / 1e9
print(f"Modèle chargé — VRAM au repos, poids seuls : {VRAM_POIDS_GO:.2f} Go")
print("(le chiffre à reporter est le PIC mesuré plus bas, pas celui-ci)")

C:\Users\theor\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


GPU : NVIDIA GeForce RTX 4060 Laptop GPU (8.6 Go VRAM)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Modèle chargé — VRAM au repos, poids seuls : 5.91 Go
(le chiffre à reporter est le PIC mesuré plus bas, pas celui-ci)


## 4. Prompt et inférence

Repris **à l'identique** du prototype : prompt, règles anti-hallucination,
parsing tolérant et retry sur sortie dégénérée. Ne pas les modifier ici sans
modifier aussi le prototype, sinon les mesures ne décrivent plus le même système.

In [4]:
_RULES = (
    "RÈGLES STRICTES — à respecter impérativement :\n"
    "1. Copie EXACTEMENT le texte imprimé sur la carte, lettre par lettre, "
    "chiffre par chiffre. Ne corrige JAMAIS l'orthographe même si un mot "
    "te semble inhabituel (ex: 'Fantyrm', 'Poissirène' sont des vrais noms).\n"
    "2. N'INVENTE JAMAIS de numéro. Si tu ne lis pas clairement les chiffres "
    "du set_number, mets null. Ne devine pas à partir du numéro d'une autre "
    "carte. Des numéros consécutifs ne sont PAS garantis.\n"
    "3. Si un champ est illisible ou partiellement caché, mets null. "
    "Un null est TOUJOURS préférable à une réponse inventée.\n"
    "4. Le set_code inclut le suffixe de langue imprimé (ex: TWMFR, SVIFR, "
    "PALFR). Ne le retire pas.\n"
    "5. ATTENTION — NE PAS confondre :\n"
    "   - Le set_number recherché est au format XXX/YYY (deux nombres séparés "
    "par un slash, ex: '067/167') imprimé EN BAS À GAUCHE de la carte, sous "
    "le texte des attaques, à côté du set_code.\n"
    "   - Il ne faut PAS prendre le 'N° XXXX' du Pokédex (ex: 'N° 0404') qui "
    "apparaît dans la bande de description sous l'illustration. Ce n'est pas "
    "le set_number.\n"
    "   - Il ne faut PAS prendre les PV (ex: 'PV 90'), ni les dégâts d'attaque "
    "(ex: '60', '40'), ni le numéro d'illustrateur.\n"
)

PROMPT_SINGLE = (
    "Cette image contient UNE carte Pokémon. Extrais ces 3 champs :\n"
    "- name : le nom du Pokémon, grand texte en haut de la carte (ex: Luxio, "
    "Fantyrm). PAS le nom d'une attaque (Grosse Morsure, Psyko, etc.).\n"
    "- set_code : le code du set imprimé en bas à gauche (ex: TWMFR, "
    "PALFR). C'est un code court de 3-5 lettres majuscules + suffixe de langue.\n"
    "- set_number : le numéro au format XXX/YYY imprimé en bas à gauche, "
    "à côté ou sous le set_code (ex: 067/167, 128/167).\n\n"
    + _RULES +
    "\nRéponds UNIQUEMENT en JSON, sans texte autour, sans markdown :\n"
    '{"name": "...", "set_code": "...", "set_number": "..."}'
)


@torch.inference_mode()
def run_vlm(image, prompt, max_new_tokens=256, do_sample=False, temperature=1.0):
    """Appelle Qwen2.5-VL et renvoie la sortie textuelle brute."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(DEVICE)

    gen_kwargs = {"max_new_tokens": max_new_tokens, "do_sample": do_sample}
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.9

    generated = model.generate(**inputs, **gen_kwargs)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


def parse_json(text):
    """Parse la sortie qui DEVRAIT etre du JSON pur. Tolerant aux fences."""
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip())
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(cleaned[start:end + 1])
        except json.JSONDecodeError:
            pass
    return {}


def is_degenerate(raw, parsed):
    """Parsing rate, ou boucle de caractere unique ('!!!!!', '.....')."""
    if not isinstance(parsed, dict) or not parsed:
        return True
    stripped = re.sub(r"\s+", "", raw or "")
    if len(stripped) >= 10:
        most_common = max(set(stripped), key=stripped.count)
        if (not most_common.isalnum()
                and stripped.count(most_common) / len(stripped) > 0.7):
            return True
    return False


print("Prompt et helpers chargés.")

Prompt et helpers chargés.


## 5. Chargement du corrigé

Trois états possibles pour un champ du corrigé :

| Valeur dans `ground_truth.json` | Signification | Effet sur les métriques |
|---|---|---|
| `"Luxio"` | la vraie valeur, lue sur la carte | comptée dans la précision |
| `null` | le champ est **réellement illisible** sur la photo | sert à détecter les hallucinations |
| `""` | **pas encore annoté** | ignoré, hors barème |

In [9]:
with open(GROUND_TRUTH, encoding="utf-8") as f:
    gt = json.load(f)

CHAMPS = ("name", "set_code", "set_number")


# Ecrit a la main : on tolere la chaine "null" au lieu du null JSON, mais on
# la signale, sinon elle passerait pour une vraie valeur et fausserait les
# comptages sans rien dire.
_FAUX_NULLS = ("null", "none", "n/a", "nul", "vide")


def etat_gt(v):
    """'vide' = pas annote · 'illisible' = null explicite · 'valeur' = annote."""
    if v is None:
        return "illisible"
    s = str(v).strip()
    if s == "":
        return "vide"
    if s.lower() in _FAUX_NULLS:
        return "illisible"
    return "valeur"


def _signaler_faux_nulls(corrige):
    """Liste les champs ecrits "null" (chaine) au lieu de null (JSON)."""
    trouves = [
        (f, c) for f, champs in corrige.items() for c in CHAMPS
        if isinstance(champs.get(c), str)
        and champs[c].strip().lower() in _FAUX_NULLS
    ]
    if trouves:
        print(
            f"\n!! {len(trouves)} champ(s) ecrits en TEXTE au lieu du null JSON.\n"
            "   Ils sont traites comme illisibles, mais corrige-les :\n"
            "   \"set_code\": null      et non   \"set_code\": \"null\""
        )
        for f, c in trouves:
            print(f"     - {f} -> {c}")
    return trouves


_signaler_faux_nulls(gt)

annotees = sorted(
    nom for nom, champs in gt.items()
    if any(etat_gt(champs.get(c)) != "vide" for c in CHAMPS)
)
restantes = sorted(set(gt) - set(annotees))

MESURE_PRECISION = bool(annotees)

print(f"Cartes dans le corrigé : {len(gt)}")
print(f"  annotées             : {len(annotees)}")
print(f"  restant à annoter    : {len(restantes)}")

if not MESURE_PRECISION:
    print(
        "\n!! Corrigé entièrement vide.\n"
        "   Le benchmark va tourner sur toutes les images et mesurer le temps\n"
        "   et la VRAM, mais la PRÉCISION restera incalculable.\n"
        "   Remplis ground_truth.json pour obtenir le critère le plus important."
    )
elif restantes:
    apercu = ", ".join(restantes[:5]) + (" ..." if len(restantes) > 5 else "")
    print(f"\n  Non annotées, donc ignorées : {apercu}")

Cartes dans le corrigé : 15
  annotées             : 15
  restant à annoter    : 0


## 6. Benchmark

Une passe sur chaque carte. Le chronomètre englobe le retry éventuel : c'est
bien le coût réel d'une carte, pas celui du cas favorable.

In [6]:
a_traiter = annotees if MESURE_PRECISION else sorted(
    p.name for p in IMAGES_DIR.glob("*.png")
)

resultats = []
vram_pic_global = 0.0

for i, fichier in enumerate(a_traiter, 1):
    chemin = IMAGES_DIR / fichier
    if not chemin.exists():
        print(f"  [{i}/{len(a_traiter)}] {fichier} — INTROUVABLE, ignorée")
        continue

    image = Image.open(chemin).convert("RGB")

    # Compteur remis à zéro juste avant : on veut le pic de CETTE inférence,
    # pas celui hérité du chargement du modèle.
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    raw = run_vlm(image, PROMPT_SINGLE)
    parsed = parse_json(raw)

    retry = is_degenerate(raw, parsed)
    if retry:
        raw = run_vlm(image, PROMPT_SINGLE, do_sample=True, temperature=0.3)
        parsed = parse_json(raw)

    torch.cuda.synchronize()
    duree_ms = (time.perf_counter() - t0) * 1000
    vram_pic = torch.cuda.max_memory_allocated() / 1e9
    vram_pic_global = max(vram_pic_global, vram_pic)

    json_ok = isinstance(parsed, dict) and bool(parsed)

    ligne = {
        "fichier": fichier,
        "duree_ms": round(duree_ms, 1),
        "vram_pic_go": round(vram_pic, 2),
        "retry": retry,
        "json_valide": json_ok,
        "raw": raw.strip().replace("\n", " ")[:300],
    }
    for c in CHAMPS:
        ligne[f"pred_{c}"] = parsed.get(c) if json_ok else None
    resultats.append(ligne)

    marque = " — RETRY" if retry else ""
    print(f"  [{i}/{len(a_traiter)}] {fichier} — {duree_ms:.0f} ms "
          f"— pic {vram_pic:.2f} Go{marque}")

print(f"\nTerminé : {len(resultats)} cartes scannées.")
print(f"Pic VRAM maximum observé : {vram_pic_global:.2f} Go")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [1/15] 0aa7a160_original.png — 6297 ms — pic 7.08 Go
  [2/15] 2b6838d7_original.png — 4670 ms — pic 7.08 Go
  [3/15] 4ab29752_original.png — 4301 ms — pic 7.08 Go
  [4/15] 4af6727c_original.png — 4621 ms — pic 7.08 Go
  [5/15] 8f17caf4_cell_01.png — 4268 ms — pic 7.01 Go
  [6/15] 8f17caf4_cell_02.png — 4195 ms — pic 7.01 Go
  [7/15] 8f17caf4_cell_03.png — 4324 ms — pic 7.01 Go
  [8/15] 8f17caf4_cell_04.png — 4412 ms — pic 7.01 Go
  [9/15] 8f17caf4_cell_05.png — 4309 ms — pic 7.01 Go
  [10/15] 8f17caf4_cell_06.png — 4269 ms — pic 7.01 Go
  [11/15] 8f17caf4_cell_07.png — 4218 ms — pic 7.01 Go
  [12/15] 8f17caf4_cell_08.png — 4404 ms — pic 7.01 Go
  [13/15] 8f17caf4_cell_09.png — 4397 ms — pic 7.01 Go
  [14/15] cdc2405a_original.png — 4791 ms — pic 7.08 Go
  [15/15] f369c2f6_original.png — 4609 ms — pic 7.08 Go

Terminé : 15 cartes scannées.
Pic VRAM maximum observé : 7.08 Go


## 7. Calcul des métriques

Définitions tirées de `docs/AI/model_evaluation/evaluation.md` §2.

In [10]:
try:
    from jiwer import cer as _cer
    HAS_JIWER = True
except ImportError:
    HAS_JIWER = False
    print("jiwer absent — CER non calculé.  pip install jiwer")


# Le prompt impose au modele un set_code sans espace ("TWMFR"), alors que la
# carte l'imprime parfois espace ("TWM FR"). Comparer strictement ferait
# echouer tous les set_code sur une convention d'ecriture, pas sur une erreur
# de lecture. Passer a True pour exiger l'egalite caractere pour caractere.
COMPARAISON_STRICTE = False


def normalise(v, champ=None):
    """Tolerant a la casse et aux espaces de bord, PAS aux accents ni aux typos.
    'luxio ' == 'Luxio'  ·  'Luxio' != 'Luxyo'

    Hors mode strict, les espaces INTERNES du set_code sont ignores :
    'TWM FR' == 'TWMFR'. Les caracteres doivent rester les memes."""
    if v is None:
        return None
    s = str(v).strip()
    if not s or s.lower() in ("null", "none", "n/a"):
        return None
    if champ == "set_code" and not COMPARAISON_STRICTE:
        s = re.sub(r"\s+", "", s)
    return s.casefold()


# --- Exact Match, champ par champ -------------------------------------------
em_par_champ, detail_em = {}, {}
for champ in CHAMPS:
    total = corrects = 0
    for r in resultats:
        brut = gt.get(r["fichier"], {}).get(champ)
        if etat_gt(brut) != "valeur":
            continue                      # non annoté ou illisible : hors barème
        total += 1
        if normalise(r[f"pred_{champ}"], champ) == normalise(brut, champ):
            corrects += 1
    em_par_champ[champ] = (corrects / total) if total else None
    detail_em[champ] = (corrects, total)

# --- Exact Match sur les champs critiques (identification de la carte) -------
crit_total = crit_ok = 0
for r in resultats:
    notables = [c for c in ("name", "set_number")
                if etat_gt(gt.get(r["fichier"], {}).get(c)) == "valeur"]
    if not notables:
        continue
    crit_total += 1
    if all(normalise(r[f"pred_{c}"], c) == normalise(gt[r["fichier"]][c], c)
           for c in notables):
        crit_ok += 1
em_critiques = (crit_ok / crit_total) if crit_total else None

# --- Field Coverage : champs attendus effectivement renseignés --------------
cov_total = cov_ok = 0
for r in resultats:
    for champ in CHAMPS:
        if etat_gt(gt.get(r["fichier"], {}).get(champ)) != "valeur":
            continue
        cov_total += 1
        if normalise(r[f"pred_{champ}"], champ) is not None:
            cov_ok += 1
coverage = (cov_ok / cov_total) if cov_total else None

# --- Hallucination : le modèle répond là où la carte est illisible ----------
hal_total = hal_ko = 0
for r in resultats:
    for champ in CHAMPS:
        if etat_gt(gt.get(r["fichier"], {}).get(champ)) != "illisible":
            continue
        hal_total += 1
        if normalise(r[f"pred_{champ}"], champ) is not None:
            hal_ko += 1
hallucination = (hal_ko / hal_total) if hal_total else None

# --- JSON Validity ----------------------------------------------------------
json_validity = (sum(r["json_valide"] for r in resultats) / len(resultats)
                 if resultats else None)
taux_retry = (sum(r["retry"] for r in resultats) / len(resultats)
              if resultats else None)

# --- CER --------------------------------------------------------------------
refs, hyps = [], []
for r in resultats:
    for champ in CHAMPS:
        brut = gt.get(r["fichier"], {}).get(champ)
        if etat_gt(brut) != "valeur":
            continue
        refs.append(str(brut).strip())
        hyps.append(str(r[f"pred_{champ}"] or "").strip())
cer_global = _cer(refs, hyps) if (HAS_JIWER and refs) else None

# --- Latence ----------------------------------------------------------------
durees = sorted(r["duree_ms"] for r in resultats)
lat_median = statistics.median(durees) if durees else None
lat_moy = statistics.mean(durees) if durees else None
lat_p95 = durees[min(int(len(durees) * 0.95), len(durees) - 1)] if durees else None

print("Métriques calculées.")

Métriques calculées.


## 8. Résultats et verdict

Seuils repris de `docs/AI/model_evaluation/evaluation.md` §9. Un modèle est
**validé sans fine-tuning** s'il les franchit tous simultanément.

In [11]:
def pct(v):
    return "non mesuré" if v is None else f"{v * 100:.1f} %"


def ms(v):
    return "non mesuré" if v is None else f"{v:.0f} ms"


print("=" * 66)
print(f"  Qwen2.5-VL-7B-Instruct (int4 nf4) — {len(resultats)} cartes")
print("=" * 66)

print("\n  Exact Match par champ")
for champ in CHAMPS:
    ok, tot = detail_em[champ]
    print(f"    {champ:<12} {pct(em_par_champ[champ]):>12}   ({ok}/{tot})")

print("\n  Temps de scan par carte")
print(f"    médiane      {ms(lat_median):>12}")
print(f"    moyenne      {ms(lat_moy):>12}")
print(f"    P95          {ms(lat_p95):>12}")

print("\n  VRAM")
print(f"    poids seuls  {VRAM_POIDS_GO:>9.2f} Go")
print(f"    PIC mesuré   {vram_pic_global:>9.2f} Go   <- chiffre à reporter")

print("\n  Robustesse")
print(f"    retry déclenché sur {pct(taux_retry)} des cartes")

# --- Verdict ---------------------------------------------------------------
SEUILS = [
    ("Exact Match (name + set_number)", em_critiques,  0.80, ">="),
    ("Field Coverage",                  coverage,      0.90, ">="),
    ("JSON Validity Rate",              json_validity, 0.95, ">="),
    ("Hallucination Rate",              hallucination, 0.05, "<="),
    ("Latence médiane (ms)",            lat_median,    3000, "<="),
    ("CER global",                      cer_global,    0.08, "<="),
]

print("\n" + "=" * 66)
print("  Seuils de décision (evaluation.md §9)")
print("=" * 66)

verdicts = []
for libelle, valeur, seuil, sens in SEUILS:
    if valeur is None:
        statut, txt = None, "non mesuré"
    else:
        statut = valeur >= seuil if sens == ">=" else valeur <= seuil
        txt = f"{valeur:.3f}" if seuil < 10 else f"{valeur:.0f}"
    verdicts.append(statut)
    marque = "?" if statut is None else ("OK" if statut else "ECHEC")
    print(f"  [{marque:^5}] {libelle:<34} {txt:>11}  (seuil {sens} {seuil})")

mesures = [v for v in verdicts if v is not None]
print("\n" + "-" * 66)
if len(mesures) < len(SEUILS):
    print("  VERDICT : incomplet — des critères n'ont pas pu être mesurés.")
    print("            Complète le corrigé, puis relance.")
elif all(mesures):
    print("  VERDICT : tous les seuils franchis.")
    print("            -> Fine-tuning nécessaire : NON")
else:
    echecs = [SEUILS[i][0] for i, v in enumerate(verdicts) if v is False]
    print("  VERDICT : seuils non franchis -> " + ", ".join(echecs))
    print("            -> Fine-tuning nécessaire : OUI")
print("-" * 66)

# --- Export ----------------------------------------------------------------
if resultats:
    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(resultats[0].keys()))
        w.writeheader()
        w.writerows(resultats)
    print(f"\nDétail par carte écrit dans {RESULTS_CSV}")

  Qwen2.5-VL-7B-Instruct (int4 nf4) — 15 cartes

  Exact Match par champ
    name               93.3 %   (14/15)
    set_code          100.0 %   (12/12)
    set_number         73.3 %   (11/15)

  Temps de scan par carte
    médiane           4397 ms
    moyenne           4539 ms
    P95               6297 ms

  VRAM
    poids seuls       5.91 Go
    PIC mesuré        7.08 Go   <- chiffre à reporter

  Robustesse
    retry déclenché sur 0.0 % des cartes

  Seuils de décision (evaluation.md §9)
  [ECHEC] Exact Match (name + set_number)          0.667  (seuil >= 0.8)
  [ OK  ] Field Coverage                           1.000  (seuil >= 0.9)
  [ OK  ] JSON Validity Rate                       1.000  (seuil >= 0.95)
  [ECHEC] Hallucination Rate                       1.000  (seuil <= 0.05)
  [ECHEC] Latence médiane (ms)                      4397  (seuil <= 3000)
  [ECHEC] CER global                               0.093  (seuil <= 0.08)

-----------------------------------------------------------

## 9. Reporter dans le rapport

Les chiffres à recopier dans `readme.md`, section « Résultats » :

| Rubrique du readme | D'où il vient |
|---|---|
| Précision de l'OCR | `Exact Match (name + set_number)` du verdict |
| Temps de scan par carte | `médiane` de la section Temps |
| Consommation VRAM | `PIC mesuré`, **pas** `poids seuls` |
| Fine-tuning nécessaire | ligne `VERDICT` |
| Avantages / limites | à rédiger : regarde `resultats_qwen.csv`, colonnes `retry` et `raw`, pour repérer les cartes qui posent problème |

Le CSV sert aussi à justifier tes conclusions : trie par `duree_ms` pour
trouver les cas lents, filtre sur `retry == True` pour les sorties instables.